**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kim-2025"], check=True)
    os.chdir("/content/encore/entries/kim-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Kim & Shin (2025) on dandiset 000248: the OpenScope Illusion data

Paper: Kim, J. H. & Shin, H. (2025). "Neural variability structure in primary visual cortex is optimal for robust representation of visual similarity", bioRxiv doi:10.1101/2025.06.30.662469 (preprint of July 2025, the text the claims file was written from; the registry lists the later title "Intrinsic geometry of trial-to-trial variability in primary visual cortex is optimal for robust representation of visual similarity").

Dataset: DANDI 000248, "Allen Institute Openscope - Illusion project", version 0.240502.2344, streamed from the archive. The paper cites the dandiset without a version. This is the only published version, so it is pinned. The paper's code read a post-processed copy dated 2024-01-30 ("OpenScopeData_00248_v240130_postprocessed"), before this version was published on 2024-05-02; no other published version exists to compare with.

The paper uses this dandiset only for the Isomap pictures of Figures 1C, 3A, 3E and 3F and for one sentence of the Methods. Its main analyses are on the Allen Visual Coding Neuropixels data, which are tested in the second notebook of this entry (`reproduce_kim_2025_allen.ipynb`).

| Claim | Data | Tested here |
|---|---|---|
| illusion-data-link | with_labels | yes |
| illusion-fig1c-isomap-geometry | dataset | yes |
| illusion-fig3a-orthogonal-variance-example | dataset | yes |
| illusion-geometry-similar-across-datasets | dataset | yes |
| all other claims | elsewhere (Allen Visual Coding Neuropixels) | second notebook |

No result here depends on random draws (Isomap uses its dense eigensolver, and the decoding check uses unshuffled folds), so no seed is needed.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, spearmanr
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score

import kim2025 as K

T0 = time.time()
pd.set_option("display.width", 200)
NAMES = K.ILLUSION_NAMES
paths = [p for p in K.asset_paths() if p.endswith("_ogen.nwb")]
print(f"dandiset {K.DANDISET_ID} version {K.DANDISET_VERSION}: {len(K.asset_paths())} assets, "
      f"{len(paths)} session files (*_ogen.nwb)")
for p in paths:
    print(" ", p)

## Data check

Every session file is opened; the V1 units (location containing VISp and not VISpm) are kept, with a flag for the Allen quality criteria, and spikes are counted in the 400 ms from the onset of each trial of the ICwcfg1 block.

In [ ]:
sessions = {}
for p in paths:
    subj = p.split("/")[0].replace("sub-", "")
    X, lab, mask, rep = K.illusion_session(p)
    sessions[subj] = dict(X=X, labels=lab, mask=mask, rep=rep, path=p)
rows = []
for subj, s in sessions.items():
    r = s["rep"]
    rows.append(dict(subject=subj, in_code_list=subj in K.CODE_SUBJECTS,
                     presentations=r["presentations"], lead_gray=r["leading_gray"],
                     trail_gray=r["trailing_gray"], trial_slots=r["slots"],
                     nongray_off_slot=r["non_gray_outside_slots"],
                     first_slot=r["first_slot"], blank_trials=r["counts_per_code"].get(0),
                     n_codes_400=len(r["codes_with_400"]), units=r["units_total"],
                     v1_units=r["v1_units"], v1_quality=r["v1_quality"],
                     v1_good_label=r["v1_good_label"], first_onset_s=round(r["first_onset"], 2),
                     total_spikes=r["total_spikes"],
                     dur_min=round(r["duration_range_s"][0], 4), dur_max=round(r["duration_range_s"][1], 4)))
summary = pd.DataFrame(rows).set_index("subject")
print(summary.to_string())
print("\nrepeats per image code (example session):", sessions[K.EXAMPLE_SUBJECT]["rep"]["counts_per_code"])
print("codes with 400 repeats in every session:",
      sorted(set.intersection(*[set(s["rep"]["codes_with_400"]) for s in sessions.values()])))
print("V1 locations, example session:", sessions[K.EXAMPLE_SUBJECT]["rep"]["v1_locations"])
allX = np.concatenate([s["X"].ravel() for s in sessions.values()])
print(f"spike counts per 400 ms trial: min {allX.min():.0f}, max {allX.max():.0f}, mean {allX.mean():.2f}")
print("first V1 unit's spike times sorted, per session:", {k: s["rep"]["first_unit_sorted"] for k, s in sessions.items()})
print("duplicate check: distinct total spike counts", summary.total_spikes.nunique(), "of", len(summary),
      "; distinct first onsets", summary.first_onset_s.nunique())

Clock check. The gray inter-trial frame shows the same image as the Blank stimulus (template 0, four disks), so a Blank trial changes nothing on the screen while the other images do. If spikes and the trial table share a clock, the V1 population rate should rise some tens of ms after the onset of image trials and not after Blank trials, and the image identity should be decodable from counts in the 400 ms after onset and not from counts in the 400 ms before it (the previous inter-trial frame).

In [ ]:
bins, ps = K.illusion_psth(sessions[K.EXAMPLE_SUBJECT]["path"])
centers = (bins[:-1] + bins[1:]) / 2
base = ps["image"][centers < 0].mean()
print(f"example session, rate (spikes/s/unit) before onset {base:.2f}; peak after image onset "
      f"{ps['image'].max():.2f} at {1000 * centers[ps['image'].argmax()]:.0f} ms; Blank trials peak "
      f"{ps['blank'][(centers > 0) & (centers < 0.2)].max():.2f}")
lat = centers[(centers > 0) & (ps["image"] > base + 0.5 * (ps["image"].max() - base))][0]
print(f"first 10 ms bin above half the peak response: {1000 * lat:.0f} ms after onset")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(1000 * centers, ps["image"], label="image trials")
ax.plot(1000 * centers, ps["blank"], label="Blank trials")
ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("ms from onset"); ax.set_ylabel("spikes/s per unit"); ax.legend()
print(f"plotted range: image {ps['image'].min():.2f} to {ps['image'].max():.2f}, blank {ps['blank'].min():.2f} to {ps['blank'].max():.2f}")
plt.show()

In [ ]:
def decode_shift(path, shift):
    f = K.open_nwb(path)
    units = K.units_table(f)
    rows_ = np.where(K.is_v1(units.location))[0]
    onsets, codes, _ = K.illusion_trials(f)
    keep = np.isin(codes, sessions[K.EXAMPLE_SUBJECT]["rep"]["codes_with_400"])
    C = K.spike_counts(f, rows_, onsets[keep] + shift, K.ILLUSION_WINDOW)
    f.close()
    cv = StratifiedKFold(5, shuffle=False)
    return cross_val_score(LinearDiscriminantAnalysis(), C, codes[keep], cv=cv).mean()

for shift in (0.0, -0.4):
    print(f"window shifted by {shift:+.1f} s: 12-way LDA accuracy {decode_shift(sessions[K.EXAMPLE_SUBJECT]['path'], shift):.3f} (chance {1/12:.3f})")

## illusion-data-link

> Dandiset 000248 holds the OpenScope Illusion recordings the paper used for its Isomap pictures, an ICwcfg1 block in which stimuli are repeated 400 times, with V1 units, in the example session the paper's code uses.

The dandiset gives the image codes 0 to 21 and an image template for each, but no names. The names come from the code's map (claims file header). The templates are drawn below so that the map can be checked against what each image shows.

In [ ]:
f = K.open_nwb(sessions[K.EXAMPLE_SUBJECT]["path"])
g = f["stimulus/templates/ICwcfg1_presentations"]
tab = f["intervals/ICwcfg1_presentations"]
print("ICwcfg1 table description:", tab.attrs.get("description"))
print("stimulus_name values:", sorted(set(K._str(tab["stimulus_name"][:]))))
print("columns:", list(tab.keys()))
fig, axes = plt.subplots(4, 6, figsize=(15, 6))
for k in range(22):
    im = g[f"ICwcfg1_presentations{k}"][::8, ::8]
    axes.flat[k].imshow(im, cmap="gray", vmin=0, vmax=255)
    axes.flat[k].set_title(f"{k}: {NAMES[k]}", fontsize=9)
for a in axes.flat:
    a.axis("off")
plt.tight_layout(); plt.show()
f.close()
ex = sessions[K.EXAMPLE_SUBJECT]
print("(i) sub-630506 present:", K.EXAMPLE_SUBJECT in sessions,
      "| ICwcfg1 block present | stimuli with 400 repeats:", len(ex["rep"]["codes_with_400"]),
      [NAMES[c] for c in ex["rep"]["codes_with_400"]])
print("(ii) Ic1, Lc1, Tre1, Tre2 among them:",
      all(c in ex["rep"]["codes_with_400"] for c in (3, 4, 10, 11)), "(codes 3, 4, 10, 11 by the code's map)")
print("(iii) V1 units in sub-630506:", ex["rep"]["v1_units"], "; passing the Allen quality criteria:", ex["rep"]["v1_quality"])
print("code's 11 subjects present:", [s for s in K.CODE_SUBJECTS if s in sessions], "missing:", [s for s in K.CODE_SUBJECTS if s not in sessions])
print("sessions in the dandiset but not in the code's list:", [s for s in sessions if s not in K.CODE_SUBJECTS])
print("400-repeat stimuli per session:", {k: len(s["rep"]["codes_with_400"]) for k, s in sessions.items()})

**Result: reproduced.** Session sub-630506 is in the dandiset with an ICwcfg1 block whose 12 stimuli with 400 repeats are the 11 image codes 1 to 7 and 10 to 13 and the Blank trials (code 0 in a trial slot), as the code expects; Ic1, Lc1, Tre1 and Tre2 (codes 3, 4, 10, 11) are among them; and it has 75 V1 units (22 by the Allen quality criteria). The dandiset names no code: the names come from the code's map, and the templates drawn above agree with it in kind (codes 8 to 13 carry a real bar, 14 to 21 a single inducer at the named corner, pointing in or out). All 11 subjects of the code are present, and a twelfth session (sub-619293) is in the dandiset but not in the code's list. Every session has 400 Blank trials and 12 stimuli with 400 repeats.

## illusion-fig1c-isomap-geometry

> In an example session of the Illusion data, three stimulus manifolds of V1 activity appear continuous with little overlap in the as-is data, discontinuous at slope 0 and heavily overlapping at slope 2.

Session sub-630506, the 12 stimuli with 400 repeats, slopes changed by residual rescaling (M2) after an M1 fit on these stimuli. Among Ic1, Lc1 and Tre1: (i) at slope 0 not one strongly connected component, (ii) as-is one, (iii) mean of the six directed overlaps larger at slope 2 than as-is, and as-is than at slope 0. Readings: the full N-neuron space and the 2D Isomap coordinates (claims file); the overlap threshold over distinct pairs (Methods, main) and over the full self-distance matrix with `<=` (code, variant); units all VISp (O3 a, main) and with the Allen quality criteria (O3 b). Test (i) is computed on the graph among the three stimuli, as "among" in the criterion reads; `trio_one_scc_via_12` also shows whether the three lie in one strongly connected component of the graph over all 12 stimuli, where a path may pass through the others.

In [ ]:
TRIO = (3, 4, 10)  # Ic1, Lc1, Tre1

def conditions(X, lab, slopes=(0, 1, 2)):
    fits = K.fit_slopes(X, lab)
    out = {"as-is": X}
    for s in slopes:
        out[f"slope {s}"] = K.manipulate(X, lab, s, fits)
    return out, fits

def trio_tests(conds, lab, space, variant):
    res = {}
    embeds = {}
    for name, Y in conds.items():
        if space == "isomap2d":
            E = K.isomap_embed(Y, 2)
            embeds[name] = E
            Y = E
        O, stims = K.overlap_matrix(Y, lab, variant)
        nodes = [list(stims).index(c) for c in TRIO]
        res[name] = dict(scc_trio=K.largest_scc(O, nodes), trio_one_scc_via_12=K.in_one_scc(O, nodes),
                         mean_overlap_trio=K.mean_offdiag(O, nodes),
                         scc_all12=K.largest_scc(O), mean_overlap_all12=K.mean_offdiag(O))
    r = pd.DataFrame(res).T
    i_ = r.loc["slope 0", "scc_trio"] < 3
    ii = r.loc["as-is", "scc_trio"] == 3
    iii = r.loc["slope 2", "mean_overlap_trio"] > r.loc["as-is", "mean_overlap_trio"] > r.loc["slope 0", "mean_overlap_trio"]
    return r, (bool(i_), bool(ii), bool(iii)), embeds

fig1c = {}
for units in ("all VISp", "quality"):
    X = ex["X"] if units == "all VISp" else ex["X"][:, ex["mask"]]
    conds, fits = conditions(X, ex["labels"])
    print(f"--- units: {units} ({X.shape[1]}); M1 slopes of the 12 stimuli {fits.slope.min():.3f} to {fits.slope.max():.3f}")
    for space in ("N-space", "isomap2d"):
        for variant in ("methods", "code"):
            r, tests, emb = trio_tests(conds, ex["labels"], space, variant)
            fig1c[(units, space, variant)] = (r, tests, emb)
            print(f"[{space}, threshold {variant}] tests (i, ii, iii) = {tests}, {sum(tests)} of 3")
            print(r.round(4).to_string())

In [ ]:
r, tests, emb = fig1c[("all VISp", "isomap2d", "methods")]
cols = {3: "tab:green", 4: "tab:orange", 10: "tab:blue"}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
lab = ex["labels"]
for a, (name, E) in zip(axes, emb.items()):
    for c in TRIO:
        a.scatter(E[lab == c, 0], E[lab == c, 1], s=3, c=cols[c], label=NAMES[c])
    a.set_title(name); a.set_aspect("equal")
    print(f"{name}: Isomap ranges x {E[:, 0].min():.1f} to {E[:, 0].max():.1f}, y {E[:, 1].min():.1f} to {E[:, 1].max():.1f}")
axes[0].legend(markerscale=4)
plt.tight_layout(); plt.show()

Other sessions (variant): the same three tests in the N-neuron space, Methods threshold, all VISp units.

In [ ]:
rows = []
for subj, s in sessions.items():
    conds, _ = conditions(s["X"], s["labels"])
    r, tests, _ = trio_tests(conds, s["labels"], "N-space", "methods")
    rows.append(dict(subject=subj, units=s["X"].shape[1], i=tests[0], ii=tests[1], iii=tests[2],
                     slope0_trio_one_scc_via_12=bool(r.loc["slope 0", "trio_one_scc_via_12"]),
                     **{f"ov_{k}": round(v, 3) for k, v in r.mean_overlap_trio.items()},
                     **{f"scc12_{k}": v for k, v in r.scc_all12.items()}))
other = pd.DataFrame(rows).set_index("subject")
print(other.to_string())
print("sessions with all three tests holding:", int((other.i & other.ii & other.iii).sum()), "of", len(other))

**Result: reproduced.** Among Ic1, Lc1 and Tre1 in sub-630506, at slope 0 the largest strongly connected component has 1 member (no overlap between the three), as-is it has all 3, and the mean directed overlap is 0.000 at slope 0, 0.221 as-is and 0.318 at slope 2. All three tests hold in every reading: the N-neuron space and the 2D Isomap coordinates, the Methods and the code thresholds, all VISp units and the units passing the quality criteria (in the 2D Isomap of the quality units the slope-0 component has 2 members, still fewer than 3). Read on the graph over all 12 stimuli, where a path may pass through the others, the three also do not lie in one strongly connected component at slope 0 in this session, under every reading (`trio_one_scc_via_12` False). Over the other sessions (variant), all three tests hold in 9 of 12; in the other 3 the three stimuli still overlap at slope 0, while the overlap rises with slope in all 12.

## illusion-fig3a-orthogonal-variance-example

> In the example session of the Illusion data, the Lc1 manifold has larger orthogonal variance toward the faraway Ic1 manifold than toward the nearby Tre2 manifold.

As-is data of sub-630506, the 12 stimuli. Geodesic distance between centroids by M5; orthogonal variance as the Methods define it.

In [ ]:
LC1, IC1, TRE2 = 4, 3, 11
for units in ("all VISp", "quality"):
    X = ex["X"] if units == "all VISp" else ex["X"][:, ex["mask"]]
    D, stims = K.centroid_geodesics(X, ex["labels"])
    st = list(stims)
    g_ic, g_tre = D[st.index(LC1), st.index(IC1)], D[st.index(LC1), st.index(TRE2)]
    ov_ic = K.orthogonal_variance(X, ex["labels"], LC1, IC1)
    ov_tre = K.orthogonal_variance(X, ex["labels"], LC1, TRE2)
    geo, ov = [], []
    for a in stims:
        for b in stims:
            if a != b:
                geo.append(D[st.index(a), st.index(b)]); ov.append(K.orthogonal_variance(X, ex["labels"], a, b))
    rho, p = spearmanr(geo, ov)
    print(f"units {units} ({X.shape[1]}): geodesic Lc1-Ic1 {g_ic:.2f}, Lc1-Tre2 {g_tre:.2f}; "
          f"orthogonal variance of Lc1 toward Ic1 {ov_ic:.4f}, toward Tre2 {ov_tre:.4f} (ratio {ov_ic / ov_tre:.2f})")
    print(f"   Ic1 farther: {g_ic > g_tre}; variance larger toward Ic1: {ov_ic > ov_tre}; "
          f"variant, Spearman over {len(geo)} ordered pairs rho = {rho:.3f}, p = {p:.2g}; finite geodesics {np.isfinite(D).all()}")

**Result: reproduced.** In sub-630506 the geodesic distance from Lc1 to Ic1 (60.0) is larger than to Tre2 (17.6), and the orthogonal variance of Lc1 toward Ic1 (0.0270) is larger than toward Tre2 (0.0161). The same holds with the quality units (40.1 against 25.8; 0.0436 against 0.0295). Over all 132 ordered pairs of the 12 stimuli the Spearman correlation between geodesic distance and orthogonal variance is weakly positive and not significant (0.150, p = 0.086; 0.163, p = 0.061 with the quality units).

## illusion-geometry-similar-across-datasets

> The changes in manifold geometry that come with changing the slope are qualitatively the same in the Illusion data as in the natural scenes data.

Every session with an ICwcfg1 block (12), V1 units, the 12 stimuli with 400 repeats, M2 at slopes 0 to 2. Per session, effective dimensionality averaged over stimuli and overlap (M3, Methods threshold) averaged over the 132 ordered pairs. Reproduced when the across-session mean dimensionality decreases and the mean overlap increases by the O2 reading, with the Wilcoxon signed-rank test (slopes 0 and 2 against as-is, across sessions, p < 0.05).

In [ ]:
t = time.time()
geom = {}
for units in ("all VISp", "quality"):
    dims, ovs = {}, {}
    for subj, s in sessions.items():
        X = s["X"] if units == "all VISp" else s["X"][:, s["mask"]]
        fits = K.fit_slopes(X, s["labels"])
        d, o = {"as-is": K.effective_dimensionality(X, s["labels"]).mean()}, {"as-is": K.mean_offdiag(K.overlap_matrix(X, s["labels"])[0])}
        for sl in K.TARGET_SLOPES:
            Y = K.manipulate(X, s["labels"], sl, fits)
            d[sl] = K.effective_dimensionality(Y, s["labels"]).mean()
            o[sl] = K.mean_offdiag(K.overlap_matrix(Y, s["labels"])[0])
        dims[subj], ovs[subj] = d, o
    geom[units] = (pd.DataFrame(dims).T, pd.DataFrame(ovs).T)
print(f"computed in {time.time() - t:.0f} s")

def o2(df, direction):
    m = df.mean()
    a0, a1, a2 = m[0.0], m[1.0], m[2.0]
    order = (a2 < a1 < a0) if direction == "decrease" else (a2 > a1 > a0)
    p0 = wilcoxon(df[0.0], df["as-is"]).pvalue
    p2 = wilcoxon(df[2.0], df["as-is"]).pvalue
    side0 = (m[0.0] > m["as-is"]) if direction == "decrease" else (m[0.0] < m["as-is"])
    side2 = (m[2.0] < m["as-is"]) if direction == "decrease" else (m[2.0] > m["as-is"])
    steps = np.diff(m[list(K.TARGET_SLOPES)].to_numpy())
    strict = bool(np.all(steps < 0)) if direction == "decrease" else bool(np.all(steps > 0))
    return dict(mean_slope0=a0, mean_slope1=a1, mean_slope2=a2, mean_asis=m["as-is"], order=order,
                p_slope0=p0, p_slope2=p2, direction_0_2_vs_asis=bool(side0 and side2), strict_every_step=strict,
                holds=bool(order and p0 < 0.05 and p2 < 0.05 and side0 and side2))

for units, (dim, ov) in geom.items():
    print(f"--- units {units}, n = {len(dim)} sessions")
    print("dimensionality (decrease):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in o2(dim, "decrease").items()})
    print("overlap (increase):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in o2(ov, "increase").items()})
    print("sessions where dimensionality at slope 2 < slope 0:", int((dim[2.0] < dim[0.0]).sum()),
          "; overlap at slope 2 > slope 0:", int((ov[2.0] > ov[0.0]).sum()))

dim, ov = geom["all VISp"]
print("\nper-session values, all VISp units (as-is, slopes 0, 1, 2):")
print(pd.concat({"dim": dim[["as-is", 0.0, 1.0, 2.0]], "overlap": ov[["as-is", 0.0, 1.0, 2.0]]}, axis=1).round(3).to_string())
print("\nexample session alone (variant):", dict(dim=dim.loc[K.EXAMPLE_SUBJECT, ["as-is", 0.0, 1.0, 2.0]].round(3).to_dict(),
                                           overlap=ov.loc[K.EXAMPLE_SUBJECT, ["as-is", 0.0, 1.0, 2.0]].round(3).to_dict()))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for a, df, name in ((axes[0], dim, "effective dimensionality"), (axes[1], ov, "mean overlap")):
    m = df[list(K.TARGET_SLOPES)]
    a.errorbar(K.TARGET_SLOPES, m.mean(), m.sem(), marker="o"); a.axhline(df["as-is"].mean(), color="k", ls="--", label="as-is")
    a.set_xlabel("slope"); a.set_title(name); a.legend()
    print(f"{name}: plotted means {m.mean().min():.3f} to {m.mean().max():.3f}")
plt.tight_layout(); plt.show()
print(f"notebook time so far {(time.time() - T0) / 60:.1f} min")

**Result: reproduced.** Across the 12 sessions, the mean effective dimensionality falls from 0.243 at slope 0 to 0.081 at slope 1 and 0.033 at slope 2 (as-is 0.073), and the mean overlap rises from 0.142 to 0.398 and 0.603 (as-is 0.429), in every 0.1 step; slopes 0 and 2 differ from as-is by the Wilcoxon signed-rank test (p = 0.0005 each). The directions at slopes 0, 1 and 2 against as-is hold in each of the 12 sessions (table above) and with the quality units; the step-by-step change is shown for the across-session mean only. These are the directions the paper reports for Figures 1D and 2A of the natural scenes data.

## Summary

| Claim | Outcome | Rests on |
|---|---|---|
| illusion-data-link | reproduced | the code's names for the image codes (the dandiset has none); Blank trials recovered from the trial sequence |
| illusion-fig1c-isomap-geometry | reproduced | every reading |
| illusion-fig3a-orthogonal-variance-example | reproduced | every reading |
| illusion-geometry-similar-across-datasets | reproduced | every reading |

How the analysis differs from the paper's: the paper's code read a post-processed MATLAB copy of the dandiset ("v240130") that is not public. Here the counts are made from the NWB spike times, and the Blank trials are recovered from the alternation of trials and gray inter-trial frames. The code applies no unit filter, and the quality-filtered units are run as a second reading. The claims on the 2D Isomap picture are judged with the paper's overlap and connectivity measures, as the claims file sets out.

What was found about the dataset: the ICwcfg1 table lists every 400 ms frame, including the inter-trial gray frames, which show the same image as the Blank stimulus (template 0), under one column `frame`. Nothing marks which gray frames are Blank trials, and the image codes have no names. The trial structure can be recovered: 60 leading gray frames, then trials and gray frames alternating, with 400 Blank trials in every session. The spikes share the clock of the table: the V1 response rises 75 ms after image onsets and not after Blank onsets, and the image identity is decodable from the 400 ms after onset (0.576, chance 0.083) and not from the 400 ms before (0.089). The units table describes none of its columns ("no description"). No session is deposited twice. Twelve sessions hold the block; the paper's code lists 11 and leaves out sub-619293.